In [67]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

In [68]:
import anthropic
from dotenv import load_dotenv
import pandas as pd 
import importlib
import time 
import src.fetcher
import src.filters
importlib.reload(src.fetcher)
importlib.reload(src.filters)

from src.fetcher import configure_entrez, fetch_sra_studies, fetch_runs_for_bioproject
from src.filters import check_hard_filters

load_dotenv('../.env')
configure_entrez()

api_key = os.getenv('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=api_key)

print('Setup complete. Ready to run tests.')
print('Anthropic client initialized:', client is not None)

Entrez configured with email: akharya@ucsd.edu
Setup complete. Ready to run tests.
Anthropic client initialized: True


In [64]:
# fetch the frog study 
frog_ids = fetch_sra_studies("PRJNA836960[BioProject]", max_results=200)
frog_df = None

for sra_id in frog_ids:
    runs_df = fetch_runs_for_bioproject(sra_id)
    if runs_df is not None and len(runs_df) > 0:
        if frog_df is None:
            frog_df = runs_df
        else:
            frog_df = pd.concat([frog_df, runs_df]).drop_duplicates(subset=["Run"])

filters = check_hard_filters(frog_df)

# summarize study metadata for the agent
study_summary = f"""
BioProject: PRJNA836960
Title: Shotgun versus 16S - Museum and Fresh Leopard Frog Guts
Host organism: Leopard Frog (Amphibia)
Total runs: {len(frog_df)}
Library strategies: {frog_df['LibraryStrategy'].value_counts().to_dict()}
Unique BioSamples: {frog_df['BioSample'].nunique()}
Platform: {frog_df['Platform'].iloc[0]}
Hard filter results: {filters}
"""

print(study_summary)

Found 130 studies. Fetching 130...
 Fetched 100 of 130
 Fetched 130 of 130


KeyError: 'library_strategy'

In [ ]:
# first anthropic api call - get agent to summarize and assess a study
message = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    system="""You are a microbiome research assistant helping curate studies 
for the Gut Microbiome Tree of Life project (GMToL). GMToL is building 
the most comprehensive cross-species gut microbiome dataset ever assembled, 
spanning insects, fish, amphibians, reptiles, birds, and mammals including 
diverse human populations. The project prioritizes host phylogenetic diversity 
— studies from underrepresented animal clades are more valuable than additional 
human or mouse studies. You help researchers decide which public studies are 
worth including by summarizing study metadata and asking targeted questions.""",
    messages=[
        {
            "role": "user",
            "content": f"""I found a candidate study for GMToL. Please:
1. Summarize what this study is in 2-3 sentences
2. Assess how valuable it is for GMToL and why
3. Ask me one specific question to help decide whether to include it

Here is the study metadata:
{study_summary}"""
        }
    ]
)

print(message.content[0].text)

## Study Summary

This study compares shotgun metagenomic (WGS) and 16S amplicon sequencing approaches using gut samples from leopard frogs, including both museum-preserved specimens and fresh samples. With 130 samples (112 WGS, 18 amplicon), it appears designed to evaluate how sequencing methodology and sample preservation affect microbiome characterization in amphibians.

## Value Assessment for GMToL

**High value.** Amphibians are significantly underrepresented in gut microbiome research compared to mammals, making this study valuable for GMToL's goal of phylogenetic diversity. The substantial WGS component (112 runs) is particularly useful since shotgun data enables strain-level resolution and functional profiling. The comparison of museum vs. fresh specimens could also inform GMToL's potential use of archived samples to expand host diversity.

## Clarifying Question

**Do you know if the 112 WGS samples are true shotgun metagenomes of the gut microbiome, or could they be host gen

In [ ]:
# researcher feedback loop 
print('Question from agent:')
print("Were the museum specimens' gut contents preserved in a way that maintains microbial DNA integrity (e.g., ethanol-preserved vs. formalin-fixed), and do you want to include both museum and fresh samples, or only the fresh specimens to ensure data quality consistency across GMToL?")
print('Do you want to include both museum and fresh samples, or only fresh samples')

#record researcher decision 
decision = input('Include this study? y/n/maybe').strip().lower()
notes = input('Any notes on why? (press enter to skip)').strip()

# store researcher feedback 
decisions = {}
decisions["PRJNA836960"] = {
    "title": "Shotgun versus 16S - Museum and Fresh Leopard Frog Guts",
    "host": "Leopard Frog (Amphibia)",
    "decision": decision,
    "notes": notes,
    "agent_assessment": "Moderately High Value - amphibian, paired data, museum preservation concern"
}
print(f"\nDecision recorded: {decision}")
if notes:
    print(f"Notes: {notes}")

print(f"\nTotal decisions made so far: {len(decisions)}")

Question from agent:
Were the museum specimens' gut contents preserved in a way that maintains microbial DNA integrity (e.g., ethanol-preserved vs. formalin-fixed), and do you want to include both museum and fresh samples, or only the fresh specimens to ensure data quality consistency across GMToL?
Do you want to include both museum and fresh samples, or only fresh samples

Decision recorded: yes
Notes: underrepresented host

Total decisions made so far: 1


Now, trying to make the agent loop through multiple studies at once automatically instead of just one. 

In [ ]:
import time 

def run_agent_triage(search_term, max_studies = 5, decisions = None):
    """
    Run the agent triage loop on a set of studies. 
    Fetches studies, filters them based on the hard filters, and presents the ones that pass to the researcher. 

    Args:
    search_term: what to search for in SRA
    max_studies: how many studies to evaluate 
    decisions: existing decisions dictionary to append to

    Returns:
    updated decisions dictionary 
    """

    if decisions is None:
        decisions = {}

    print(f'Searching for: {search_term}\n')
    ids = fetch_sra_studies(search_term, max_results = max_studies * 3)  # fetch extra to account for filtering since most won't pass hard filters

    studies_presented = 0
    
    for sra_id in ids:
        if studies_presented >= max_studies:
            break

        runs_df = fetch_runs_for_bioproject(sra_id)
        if runs_df is None or len(runs_df) ==0:
            continue 
        
        filters = check_hard_filters(runs_df)

        if not filters['passes']:
            continue

        bioproject = runs_df['BioProject'].iloc[0]

        if bioproject in decisions:
            print(f" Skipping {bioproject} - already evaluated")
            continue

        study_summary = f"""
BioProject: {bioproject}
Host organism: {runs_df['ScientificName'].iloc[0]}
Total runs: {len(runs_df)}
Library strategies: {runs_df['LibraryStrategy'].value_counts().to_dict()}
Unique BioSamples: {runs_df['BioSample'].nunique()}
Platform: {runs_df['Platform'].iloc[0]}
Hard filter results: {filters}
"""
        
        # ask agent to assess
        message = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=512,
            system="""You are a microbiome research assistant helping curate 
studies for the Gut Microbiome Tree of Life project (GMToL). GMToL prioritizes 
host phylogenetic diversity — studies from underrepresented animal clades like 
amphibians, reptiles, insects, and fish are more valuable than additional human 
or mouse studies. Be concise — summarize in 2-3 sentences, give a value rating 
(Low/Medium/High/Critical), and ask one specific question.""",
            messages=[
                {
                    "role": "user",
                    "content": f"Assess this candidate study for GMToL:\n{study_summary}"
                }
            ]
        )

        print(f"Study {studies_presented + 1}: {bioproject}")
        print(f"{'='*50}")
        print(message.content[0].text)
        print()
        
        # get researcher feedback
        decision = input("Include this study? (y/n/maybe/stop): ").strip().lower()
        
        if decision == "stop":
            print("Stopping triage session.")
            break
            
        notes = input("Notes (press enter to skip): ").strip()
        
        decisions[bioproject] = {
            "host": runs_df["ScientificName"].iloc[0],
            "decision": decision,
            "notes": notes,
            "strategies": runs_df["LibraryStrategy"].value_counts().to_dict()
        }
        
        studies_presented += 1
        print(f"Decision recorded. ({studies_presented} studies reviewed)\n")
        time.sleep(0.5)
    
    return decisions

# run the agent on paired gut microbiome studies
all_decisions = run_agent_triage(
    search_term="gut microbiome 16S metagenomics",
    max_studies=3,
    decisions=decisions  # pass in existing decisions so we don't repeat
)

print(f"SESSION COMPLETE")
print(f"Total decisions in log: {len(all_decisions)}")
print(f"\nDecision summary:")
for bioproject, info in all_decisions.items():
    print(f"  {bioproject} | {info['host']} | {info['decision']}")

Searching for: gut microbiome 16S metagenomics

Found 91311 studies. Fetching 9...
 Fetched 9 of 9
SESSION COMPLETE
Total decisions in log: 1

Decision summary:
  PRJNA836960 | Leopard Frog (Amphibia) | yes


In [ ]:
# try the dual strategy - search for paired studies first
all_decisions = run_agent_triage(
    search_term="16S metagenomics gut paired same samples",
    max_studies=3,
    decisions=all_decisions
)

Searching for: 16S metagenomics gut paired same samples

Found 391 studies. Fetching 9...
 Fetched 9 of 9


In [ ]:
# what's actually in those 9 studies
from Bio import Entrez

handle = Entrez.esearch(
    db="sra",
    term="16S metagenomics gut paired same samples",
    retmax=9
)
record = Entrez.read(handle)
handle.close()

diag_ids = record["IdList"]

for sra_id in diag_ids:
    runs_df = fetch_runs_for_bioproject(sra_id)
    if runs_df is not None:
        bioproject = runs_df["BioProject"].iloc[0]
        strategies = runs_df["LibraryStrategy"].unique()
        print(f"SRA ID: {sra_id} | BioProject: {bioproject} | Strategies: {strategies} | Runs: {len(runs_df)}")
    time.sleep(0.3)

SRA ID: 36740359 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22
SRA ID: 36740358 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22
SRA ID: 36740357 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22
SRA ID: 36740356 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22
SRA ID: 36740355 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22
SRA ID: 36740354 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22
SRA ID: 36740353 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22
SRA ID: 36740352 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22
SRA ID: 36740351 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22


In [ ]:
# deduplicated search (check each BioProject only once)
handle = Entrez.esearch(
    db="sra",
    term="16S metagenomics gut paired same samples",
    retmax=50
)
record = Entrez.read(handle)
handle.close()

diag_ids = record["IdList"]
seen_bioprojects = set()

print("Checking unique BioProjects...\n")

for sra_id in diag_ids:
    runs_df = fetch_runs_for_bioproject(sra_id)
    if runs_df is None:
        continue
    
    bioproject = runs_df["BioProject"].iloc[0]
    
    if bioproject in seen_bioprojects:
        continue
    seen_bioprojects.add(bioproject)
    
    strategies = runs_df["LibraryStrategy"].unique()
    filters = check_hard_filters(runs_df)
    
    print(f"BioProject: {bioproject} | Strategies: {strategies} | Runs: {len(runs_df)} | passes: {filters['passes']}")
    time.sleep(0.3)

print(f"\nUnique BioProjects checked: {len(seen_bioprojects)}")

Checking unique BioProjects...

BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22 | passes: False
BioProject: PRJNA1168565 | Strategies: ['RNA-Seq'] | Runs: 32 | passes: False

Unique BioProjects checked: 2


In [ ]:
# search BioProject database directly for projects with both data types
handle = Entrez.esearch(
    db="bioproject",
    term="gut microbiome amplicon metagenomics",
    retmax=20
)
record = Entrez.read(handle)
handle.close()

print("BioProject results:", record["Count"])
bioproject_ids = record["IdList"]
print("IDs:", bioproject_ids)

BioProject results: 35
IDs: ['1436233', '1434924', '1392086', '1390320', '1369179', '1328449', '1302757', '1283070', '1255824', '1069622', '1068958', '1060181', '1059984', '1053582', '1043851', '1043676', '979790', '918035', '860452', '832581']


In [65]:
def fetch_runs_by_bioproject_accession(bioproject_accession):
    """
    Fetch all runs for a BioProject using its accession number directly.
    
    Args:
        bioproject_accession: BioProject accession like PRJNA836960
    Returns:
        DataFrame of all runs or None if fetch failed
    """
    df = None
    
    try:
        # search SRA for all runs belonging to this BioProject
        handle = Entrez.esearch(
            db="sra",
            term=f"{bioproject_accession}[BioProject]",
            retmax=1000
        )
        record = Entrez.read(handle)
        handle.close()
        time.sleep(0.3)
        
        if record["Count"] == "0":
            return None
            
        all_ids = record["IdList"]
        ids_string = ",".join(all_ids)
        
        fetch_handle = Entrez.efetch(
            db="sra",
            id=ids_string,
            rettype="runinfo",
            retmode="text"
        )
        data = fetch_handle.read()
        fetch_handle.close()
        
        decoded = data.decode("utf-8")
        df = pd.read_csv(io.StringIO(decoded))
        df = df.dropna(subset=["Run"])
        time.sleep(0.3)
        
    except Exception as e:
        print(f"Error fetching {bioproject_accession}: {e}")
    
    return df



# fetch BioProject records to get accessions
print("Fetching BioProject accessions...\n")

fetch_handle = Entrez.efetch(
    db="bioproject",
    id=",".join(bioproject_ids),
    rettype="xml",
    retmode="xml"
)
bp_data = fetch_handle.read()
fetch_handle.close()

print("Raw BioProject data (first 2000 chars):")
print(bp_data[:2000])

Fetching BioProject accessions...



NameError: name 'bioproject_ids' is not defined

In [66]:
import xml.etree.ElementTree as ET

# parse the XML to extract BioProject accessions
root = ET.fromstring(bp_data)

accessions = []
for doc in root.findall("DocumentSummary"):
    for archive_id in doc.findall(".//ArchiveID"):
        accession = archive_id.get("accession")
        if accession:
            accessions.append(accession)

print(f"Found {len(accessions)} BioProject accessions:")
for acc in accessions:
    print(f"  {acc}")

NameError: name 'bp_data' is not defined

In [ ]:
import io
print("Checking hard filters on 20 BioProjects...\n")

bioproject_results = []

for accession in accessions:
    runs_df = fetch_runs_by_bioproject_accession(accession)
    
    if runs_df is None or len(runs_df) == 0:
        print(f"{accession}: no runs found")
        continue
    
    filters = check_hard_filters(runs_df)
    strategies = runs_df["LibraryStrategy"].unique().tolist()
    host = runs_df["ScientificName"].iloc[0]
    
    print(f"{accession} | {host[:30]} | strategies: {strategies} | passes: {filters['passes']}")
    
    bioproject_results.append({
        "accession": accession,
        "host": host,
        "strategies": strategies,
        "filters": filters,
        "runs": len(runs_df)
    })
    
    time.sleep(0.5)

# summary
passing = [r for r in bioproject_results if r["filters"]["passes"]]
print(f"\n--- Summary ---")
print(f"Projects checked: {len(bioproject_results)}")
print(f"Projects passing all filters: {len(passing)}")
if passing:
    print("\nPassing projects:")
    for p in passing:
        print(f"  {p['accession']} | {p['host']} | {p['strategies']}")

Checking hard filters on 20 BioProjects...



NameError: name 'fetch_runs_by_bioproject_accession' is not defined

In [ ]:
from src.decisions import save_decisions, load_decisions

# run agent on the two passing studies
for result in passing:
    study_summary = f"""
BioProject: {result['accession']}
Host organism: {result['host']}
Total runs: {result['runs']}
Library strategies: {result['strategies']}
Hard filter results: {result['filters']}
"""
    
    message = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=512,
        system="""You are a microbiome research assistant helping curate 
studies for the Gut Microbiome Tree of Life project (GMToL). GMToL prioritizes 
host phylogenetic diversity — studies from underrepresented animal clades are 
more valuable than additional human or mouse studies. Be concise — summarize 
in 2-3 sentences, give a value rating (Low/Medium/High/Critical), and ask 
one specific question.""",
        messages=[
            {
                "role": "user",
                "content": f"Assess this candidate study for GMToL:\n{study_summary}"
            }
        ]
    )
    
    
    print(f"Study: {result['accession']} | Host: {result['host']}")
    
    print(message.content[0].text)
    
    decision = input("\nInclude this study? (y/n/maybe): ").strip().lower()
    notes = input("Notes (press enter to skip): ").strip()
    
    all_decisions[result['accession']] = {
        "host": result['host'],
        "decision": decision,
        "notes": notes,
        "strategies": result['strategies']
    }
    
    print(f"Decision recorded.\n")
    time.sleep(0.5)

    all_decisions = load_decisions()
    print(f"Loaded {len(all_decisions)} existing decisions")
for bp, info in all_decisions.items(): 
    print(f"  {bp} | {info['host']} | {info['decision']}")

NameError: name 'passing' is not defined

In [ ]:
from src.ena_fetcher import fetch_runs_for_study, resolve_host_species,\
fetch_pubmed_id, fetch_pubmed_abstract, fetch_study_origin
from src.fetcher import configure_entrez

configure_entrez()
#1. obtaining all the info to verify from abstract
runs_df = fetch_runs_for_study("PRJNA836960")
host_species = resolve_host_species(runs_df)
print("Host species:", host_species)
lib_strategies = runs_df['library_strategy'].value_counts().to_dict()
print("Library strategies:", lib_strategies)
origin = fetch_study_origin("PRJNA836960")
title = origin.get('title')
print("Study title:", title)

#2. get abstract from PubMed

abstract = fetch_pubmed_abstract("PRJNA836960")
print("Abstract:", abstract)

#3. build prompt string for agent
prompt = f"""
Study title: {title}
Host species: {host_species}
Library strategies: {lib_strategies}
Abstract: {abstract}
"""

#4. Claude API call to check consistency of metadata with abstract
message = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=512,
    system="You are a scientific quality control assistant checking metadata consistency for microbiome studies.",
    messages=[
        {
            "role": "user",
            "content": prompt + "\n\nPlease check: 1) Does host species match the abstract? 2) Do sequencing methods match? 3) Is this a gut microbiome study? 4) Any red flags? Rate consistency: High/Medium/Low"
        }
 
   ]
)

print(message.content[0].text)

Entrez configured with email: akharya@ucsd.edu
Host species: Lithobates pipiens
Library strategies: {'WGS': 112, 'AMPLICON': 18}
Study title: None
Abstract: None
# Quality Control Assessment

## 1. Host Species Match with Abstract
**Cannot assess** - No abstract provided (None)

## 2. Sequencing Methods Consistency
**Potential concern**: Mixed library strategies detected
- WGS (Whole Genome Sequencing): 112 samples
- AMPLICON: 18 samples

This combination could be legitimate (e.g., WGS for metagenomics + 16S amplicon for taxonomic profiling), but warrants verification that these are intentionally different approaches within the same study.

## 3. Gut Microbiome Study?
**Cannot confirm** - No abstract or title provided to determine study focus. *Lithobates pipiens* (Northern Leopard Frog) is a common model organism for amphibian microbiome research, including gut, skin, and environmental microbiome studies.

## 4. Red Flags
- ⚠️ **Missing title** (None)
- ⚠️ **Missing abstract** (None)


In [ ]:
provenance = fetch_study_origin("PRJNA836960")
print(provenance)

{'accession': 'PRJNA836960', 'source': 'NCBI', 'title': None, 'description': None}


In [ ]:
# debug step 1 - check if NCBI BioProject search works
from Bio import Entrez

handle = Entrez.esearch(
    db="bioproject",
    term="PRJNA836960[Project Accession]",
    retmax=1
)
record = Entrez.read(handle)
handle.close()
print("IDs found:", record["IdList"])
print("Count:", record["Count"])

IDs found: ['836960']
Count: 1


In [ ]:
# debug step 2 - fetch and print raw XML
fetch_handle = Entrez.efetch(
    db="bioproject",
    id="836960",
    rettype="xml",
    retmode="xml"
)
data = fetch_handle.read()
fetch_handle.close()
print(data[:2000])

b'<?xml version="1.0" ?>\n<RecordSet><DocumentSummary uid="836960">\n    <Project>\n        <ProjectID>\n            <ArchiveID accession="PRJNA836960" archive="NCBI" id="836960"/>\n            <LocalID>bp0</LocalID>\n            <LocalID>bp0</LocalID>\n        </ProjectID>\n        <ProjectDescr>\n            <Title>Gut Microbiota of Amphibian Museum Specimens</Title>\n            <Description>The study goal was to examine the ecological and evolutionary changes associated with gut microbiota of amphibian assemblages. This was done with fluid-preserved museum specimens derived from habitats with spatiotemporal differences in land-use.</Description>\n            <Grant GrantId="1907311">\n                <Title>Host-Microbiome Ecology</Title>\n                <Agency abbr="NSF">National Science Foundation</Agency>\n            </Grant>\n            <Relevance>\n                <Evolution>yes</Evolution>\n            </Relevance>\n        </ProjectDescr>\n        <ProjectType>\n        

title exists but is nested inside <ProjectDescr>

In [ ]:
provenance = fetch_study_origin("PRJNA836960")
print(provenance)

{'accession': 'PRJNA836960', 'source': 'NCBI', 'title': None, 'description': None}


In [ ]:
import xml.etree.ElementTree as ET

# use the raw XML we already fetched
root = ET.fromstring(data)

# try different paths
print("Path 1:", root.find(".//ProjectDescr/Title"))
print("Path 2:", root.find(".//Title"))
print("Path 3:", root.find(".//{*}Title"))

# print all tags in the XML
for elem in root.iter():
    if "Title" in elem.tag or "title" in elem.tag:
        print(f"Tag: {elem.tag}, Text: {elem.text}")

Path 1: <Element 'Title' at 0x128bd4180>
Path 2: <Element 'Title' at 0x128bd4180>
Path 3: <Element 'Title' at 0x128bd4180>
Tag: Title, Text: Gut Microbiota of Amphibian Museum Specimens
Tag: Title, Text: Host-Microbiome Ecology


In [ ]:
provenance = fetch_study_origin("PRJNA836960")
print(provenance)

{'accession': 'PRJNA836960', 'source': 'NCBI', 'title': None, 'description': None}


In [ ]:
import requests

params = {
    "result": "study",
    "query": 'study_accession="PRJNA836960"',
    "fields": "study_accession,study_title,study_description",
    "limit": 1,
    "format": "json"
}

response = requests.get("https://www.ebi.ac.uk/ena/portal/api/search", params=params, timeout=30)
print("Status:", response.status_code)
print("Response:", response.text)

Status: 200
Response: [
{"study_accession":"PRJNA836960","study_title":"Gut Microbiota of Amphibian Museum Specimens","study_description":"The study goal was to examine the ecological and evolutionary changes associated with gut microbiota of amphibian assemblages. This was done with fluid-preserved museum specimens derived from habitats with spatiotemporal differences in land-use."}
]



In [ ]:
provenance = fetch_study_origin("PRJNA836960")
print(provenance)

{'accession': 'PRJNA836960', 'source': 'NCBI', 'title': None, 'description': None}


In [ ]:
import importlib
import src.ena_fetcher
importlib.reload(src.ena_fetcher)

from src.ena_fetcher import fetch_study_origin
result = fetch_study_origin("PRJNA836960")
print(result)

{'accession': 'PRJNA836960', 'source': 'NCBI', 'title': 'Gut Microbiota of Amphibian Museum Specimens', 'description': 'The study goal was to examine the ecological and evolutionary changes associated with gut microbiota of amphibian assemblages. This was done with fluid-preserved museum specimens derived from habitats with spatiotemporal differences in land-use.'}


In [ ]:
from src.ena_fetcher import fetch_runs_for_study, resolve_host_species,\
fetch_pubmed_id, fetch_pubmed_abstract, fetch_study_origin
from src.fetcher import configure_entrez

configure_entrez()
#1. obtaining all the info to verify from abstract
runs_df = fetch_runs_for_study("PRJNA836960")
host_species = resolve_host_species(runs_df)
print("Host species:", host_species)
lib_strategies = runs_df['library_strategy'].value_counts().to_dict()
print("Library strategies:", lib_strategies)
origin = fetch_study_origin("PRJNA836960")
title = origin.get('title')
print("Study title:", title)

#2. get abstract from PubMed

abstract = fetch_pubmed_abstract("PRJNA836960")
print("Abstract:", abstract)

#3. build prompt string for agent
prompt = f"""
Study title: {title}
Host species: {host_species}
Library strategies: {lib_strategies}
Abstract: {abstract}
"""

#4. Claude API call to check consistency of metadata with abstract
message = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=512,
    system="You are a scientific quality control assistant checking metadata consistency for microbiome studies.",
    messages=[
        {
            "role": "user",
            "content": prompt + "\n\nPlease check: 1) Does host species match the abstract? 2) Do sequencing methods match? 3) Is this a gut microbiome study? 4) Any red flags? Rate consistency: High/Medium/Low"
        }
 
   ]
)

print(message.content[0].text)

Entrez configured with email: akharya@ucsd.edu
Host species: Lithobates pipiens
Library strategies: {'WGS': 112, 'AMPLICON': 18}
Study title: Gut Microbiota of Amphibian Museum Specimens
Abstract: None
## Metadata Consistency Check

### 1. Host Species vs Abstract Match
**Cannot verify** - No abstract provided. However, *Lithobates pipiens* (Northern Leopard Frog) is consistent with the study title mentioning "Amphibian Museum Specimens."

### 2. Sequencing Methods Match
**Potential concern** - The study shows two library strategies:
- WGS (Whole Genome Sequencing): 112 samples
- AMPLICON: 18 samples

For a gut microbiota study, AMPLICON (typically 16S rRNA) is expected. The predominance of WGS (86%) is unusual for a standard microbiome survey, though it could indicate:
- Shotgun metagenomics approach
- Combined host + microbiome sequencing from museum specimens
- Possible metadata inconsistency

### 3. Gut Microbiome Study?
**Yes** - Title explicitly states "Gut Microbiota" study.

##

In [ ]:
from src.ena_fetcher import search_ena_studies

In [ ]:
# full pipeline loop to loop through multiple studies
# fetch runs, host species, check hard filters and origin
# store results in a list of dictionaries 

query = 'tax_tree(749906)' # board gut metagenome studies 
study_accessions = search_ena_studies(query, max_results=80)
seen = set()
results = []

for study in study_accessions:
    if study in seen:
        continue
    seen.add(study)

    runs_df = fetch_runs_for_study(study)
    if runs_df is None or len(runs_df) == 0:
        continue 
    host = resolve_host_species(runs_df)
    filters = check_hard_filters(runs_df)

    print (f" study: {study} | host: {host} | filters pass: {filters['passes']}")
    
    result = {
        "study": study,
        "host": host,
        "library_strategies": runs_df['library_strategy'].value_counts().to_dict(),
        "passes": filters['passes'], 
        "has_16S": filters['has_16S'],
        "has_shotgun": filters['has_shotgun'], 
        "is_paired": filters['is_paired']
    }
    
    if filters['passes']:
        origin = fetch_study_origin(study)
        result["title"] = origin.get('title')
        print(f" Passes - Title: {result['title']}")

    results.append(result)
    time.sleep(0.3)

print(f"\n Total unique studies found: {len(seen)}")
print(f"Studies passing filters: {sum(1 for r in results if r['passes'])}")

ENA search failed: 400


TypeError: 'NoneType' object is not iterable

In [ ]:
print(type(study_accessions))
print(study_accessions)

<class 'NoneType'>
None


In [ ]:
import requests

url = "https://www.ebi.ac.uk/ena/portal/api/search"
params = {
    "result": "read_run",
    "query": "tax_tree(749906)",
    "fields": "study_accession",
    "limit": 10,
    "format": "json"
}

response = requests.get(url, params=params, timeout=30)
print("Status:", response.status_code)
print("Response:", response.text[:500])

Status: 200
Response: [
{"run_accession":"DRR013349","study_accession":"PRJDB2173"}
,
{"run_accession":"DRR018138","study_accession":"PRJDB1856"}
,
{"run_accession":"DRR018139","study_accession":"PRJDB1856"}
,
{"run_accession":"DRR018145","study_accession":"PRJDB1856"}
,
{"run_accession":"DRR018153","study_accession":"PRJDB1856"}
,
{"run_accession":"DRR018154","study_accession":"PRJDB1856"}
,
{"run_accession":"DRR018155","study_accession":"PRJDB1856"}
,
{"run_accession":"DRR018157","study_accession":"PRJDB1856"}
,
{"


In [ ]:
import importlib
import src.ena_fetcher
importlib.reload(src.ena_fetcher)

from src.ena_fetcher import search_ena_studies
result = search_ena_studies("tax_tree(749906)", max_results=50)
print("Result:", result)

Found 500 runs, 28 unique studies
Result: ['PRJDB7910', 'PRJDB7938', 'PRJDB10554', 'PRJDB3819', 'PRJDB1856', 'PRJDB8861', 'PRJDB5181', 'PRJDB6775', 'PRJDB4366', 'PRJDB9133', 'PRJDB7203', 'PRJDB5739', 'PRJDB3161', 'PRJDB9844', 'PRJDB2173', 'PRJDB6933', 'PRJDB8266', 'PRJDB3162', 'PRJDB9535', 'PRJDB9544', 'PRJDB10519', 'PRJDB5334', 'PRJDB9608', 'PRJDB9438', 'PRJDB4237', 'PRJDB5278', 'PRJDB5748', 'PRJDB5420']


In [ ]:
# full pipeline loop to loop through multiple studies
# fetch runs, host species, check hard filters and origin
# store results in a list of dictionaries 

query = 'tax_tree(749906)' # board gut metagenome studies 
study_accessions = search_ena_studies(query, max_results=50)
seen = set()
results = []

for study in study_accessions:
    if study in seen:
        continue
    seen.add(study)

    runs_df = fetch_runs_for_study(study)
    if runs_df is None or len(runs_df) == 0:
        continue 
    host = resolve_host_species(runs_df)
    filters = check_hard_filters(runs_df)

    print (f" study: {study} | host: {host} | filters pass: {filters['passes']}")
    
    result = {
        "study": study,
        "host": host,
        "library_strategies": runs_df['library_strategy'].value_counts().to_dict(),
        "passes": filters['passes'], 
        "has_16S": filters['has_16S'],
        "has_WGS": filters['has_WGS'], 
        "is_paired": filters['is_paired']
    }
    
    if filters['passes']:
        origin = fetch_study_origin(study)
        result["title"] = origin.get('title')
        print(f" Passes - Title: {result['title']}")

    results.append(result)
    time.sleep(0.3)

print(f"\n Total unique studies found: {len(seen)}")
print(f"Studies passing filters: {sum(1 for r in results if r['passes'])}")

DEBUG: searching for tax_tree(749906), max_results=50
DEBUG: offset=0, status=200, accessions so far=0
  Got 100 runs, 9 unique studies so far
DEBUG: offset=100, status=400, accessions so far=9
ENA search failed: 400
 study: PRJDB5181 | host: Penaeus vannamei | filters pass: False
 study: PRJDB4366 | host: Apostichopus japonicus | filters pass: True
 Passes - Title: Sea cucmber gut microbiome
 study: PRJDB3161 | host: Homo sapiens | filters pass: False
 study: PRJDB2173 | host: None | filters pass: False
 study: PRJDB3819 | host: Lagopus muta hyperborea | filters pass: False
 study: PRJDB4237 | host: Apis mellifera | filters pass: False
 study: PRJDB5278 | host: Penaeus vannamei | filters pass: False
 study: PRJDB1856 | host: biofilm metagenome | filters pass: False
 study: PRJDB3162 | host: Homo sapiens | filters pass: False

 Total unique studies found: 9
Studies passing filters: 1


In [ ]:
params = {
    "result": "read_run",
    "query": "tax_tree(749906)",
    "fields": "study_accession",
    "limit": 100,
    "offset": 100,
    "format": "json"
}

response = requests.get("https://www.ebi.ac.uk/ena/portal/api/search", params=params, timeout=30)
print("Status:", response.status_code)
print("Response:", response.text[:200])

Status: 400
Response: {"message":"Unsupported param offset"}



In [ ]:
result = search_ena_studies("tax_tree(749906)", max_results=20)
print("Studies found:", len(result))
print(result)

Found 200 runs, 13 unique studies
Studies found: 13
['PRJDB5181', 'PRJDB5739', 'PRJDB4366', 'PRJDB3161', 'PRJDB5334', 'PRJDB2173', 'PRJDB3819', 'PRJDB4237', 'PRJDB5278', 'PRJDB1856', 'PRJDB3162', 'PRJDB5748', 'PRJDB5420']


In [ ]:
query = 'tax_tree(749906)' # board gut metagenome studies 
study_accessions = search_ena_studies(query, max_results=50)
seen = set()
results = []

for study in study_accessions:
    if study in seen:
        continue
    seen.add(study)

    runs_df = fetch_runs_for_study(study)
    if runs_df is None or len(runs_df) == 0:
        continue 
    host = resolve_host_species(runs_df)
    filters = check_hard_filters(runs_df)

    print (f" study: {study} | host: {host} | filters pass: {filters['passes']}")
    
    result = {
        "study": study,
        "host": host,
        "library_strategies": runs_df['library_strategy'].value_counts().to_dict(),
        "passes": filters['passes'], 
        "has_16S": filters['has_16S'],
        "has_WGS": filters['has_WGS'], 
        "is_paired": filters['is_paired']
    }
    
    if filters['passes']:
        origin = fetch_study_origin(study)
        result["title"] = origin.get('title')
        print(f" Passes - Title: {result['title']}")

    results.append(result)
    time.sleep(0.3)

print(f"\n Total unique studies found: {len(seen)}")
print(f"Studies passing filters: {sum(1 for r in results if r['passes'])}")

Found 500 runs, 28 unique studies
 study: PRJDB7910 | host: Gorilla gorilla gorilla | filters pass: False
 study: PRJDB7938 | host: Cervus nippon yakushimae | filters pass: False
 study: PRJDB10554 | host: None | filters pass: False
 study: PRJDB3819 | host: Lagopus muta hyperborea | filters pass: False
 study: PRJDB1856 | host: biofilm metagenome | filters pass: False
 study: PRJDB8861 | host: Gallus gallus | filters pass: False
 study: PRJDB5181 | host: Penaeus vannamei | filters pass: False
 study: PRJDB6775 | host: Cervus nippon yesoensis | filters pass: False
 study: PRJDB4366 | host: Apostichopus japonicus | filters pass: True
 Passes - Title: Sea cucmber gut microbiome
 study: PRJDB9133 | host: Penaeus vannamei | filters pass: False
 study: PRJDB7203 | host: Neomysis awatschensis | filters pass: False
 study: PRJDB5739 | host: Penaeus vannamei | filters pass: False
 study: PRJDB3161 | host: Homo sapiens | filters pass: False
 study: PRJDB9844 | host: Elephas maximus | filters pa

In [ ]:
import pandas as pd

summary_df = pd.DataFrame(results)
print(summary_df[["study", "host", "has_16S", "has_WGS", "is_paired", "passes"]].to_string())

         study                      host  has_16S  has_WGS  is_paired  passes
0    PRJDB7910   Gorilla gorilla gorilla     True    False      False   False
1    PRJDB7938  Cervus nippon yakushimae     True    False      False   False
2   PRJDB10554                      None     True    False      False   False
3    PRJDB3819   Lagopus muta hyperborea    False     True      False   False
4    PRJDB1856        biofilm metagenome     True    False      False   False
5    PRJDB8861             Gallus gallus     True    False      False   False
6    PRJDB5181          Penaeus vannamei     True    False      False   False
7    PRJDB6775   Cervus nippon yesoensis     True    False      False   False
8    PRJDB4366    Apostichopus japonicus     True     True       True    True
9    PRJDB9133          Penaeus vannamei     True    False      False   False
10   PRJDB7203     Neomysis awatschensis     True    False      False   False
11   PRJDB5739          Penaeus vannamei     True    False      

In [ ]:
# search multiple host groups separately
queries = [
    'tax_tree(749906)',      # gut metagenome broad
    'tax_tree(8292)',        # Amphibia
    'tax_tree(8782)',        # Aves (birds)
    'tax_tree(7898)',        # Actinopterygii (fish)
    'tax_tree(6656)',        # Arthropoda (insects, crustaceans)
    'tax_tree(7742)',        # Vertebrata
    'tax_tree(40674)',       # Mammalia
]

all_study_accessions = set()

for query in queries:
    accessions = search_ena_studies(query, max_results=20)
    all_study_accessions.update(accessions)
    print(f"Query: {query} → {len(accessions)} studies")
    time.sleep(0.5)

print(f"\nTotal unique studies across all queries: {len(all_study_accessions)}")

Found 200 runs, 13 unique studies
Query: tax_tree(749906) → 13 studies
Found 200 runs, 39 unique studies
Query: tax_tree(8292) → 20 studies
Found 200 runs, 23 unique studies
Query: tax_tree(8782) → 20 studies
Found 200 runs, 31 unique studies
Query: tax_tree(7898) → 20 studies
Found 200 runs, 53 unique studies
Query: tax_tree(6656) → 20 studies
Found 200 runs, 21 unique studies
Query: tax_tree(7742) → 20 studies
Found 200 runs, 21 unique studies
Query: tax_tree(40674) → 20 studies

Total unique studies across all queries: 112


In [ ]:
query = [
    'tax_tree(749906)',      # gut metagenome 
    'tax_tree(8292)',        # Amphibia
    'tax_tree(8782)',        # Aves (birds)
    'tax_tree(7898)',        # Actinopterygii (fish)
    'tax_tree(6656)',        # Arthropoda 
    'tax_tree(40674)',       # Mammalia
] 

all_study_accessions = set()

for query in queries:
    accessions = search_ena_studies(query, max_results=20)
    all_study_accessions.update(accessions)
    print(f"Query: {query} → {len(accessions)} studies")
    time.sleep(0.5)

print(f"\nTotal unique studies: {len(all_study_accessions)}")

seen = set()
results = []

for study in all_study_accessions:
    print(f"Processing: {study}...")
    if study in seen:
        continue
    seen.add(study)

    runs_df = fetch_runs_for_study(study)
    if runs_df is None or len(runs_df) == 0:
        continue 
    host = resolve_host_species(runs_df)
    filters = check_hard_filters(runs_df)

    print (f" study: {study} | host: {host} | filters pass: {filters['passes']}")
    
    result = {
        "study": study,
        "host": host,
        "library_strategies": runs_df['library_strategy'].value_counts().to_dict(),
        "passes": filters['passes'], 
        "has_16S": filters['has_16S'],
        "has_WGS": filters['has_WGS'], 
        "is_paired": filters['is_paired']
    }
    
    if filters['passes']:
        origin = fetch_study_origin(study)
        result["title"] = origin.get('title')
        print(f" Passes - Title: {result['title']}")

    results.append(result)
    time.sleep(0.3)

print(f"\n Total unique studies found: {len(seen)}")
print(f"Studies passing filters: {sum(1 for r in results if r['passes'])}")

print(f"Total evaluated: {len(results)}")
print(f"Total in all_study_accessions: {len(all_study_accessions)}")

Found 200 runs, 13 unique studies
Query: tax_tree(749906) → 13 studies
Found 200 runs, 39 unique studies
Query: tax_tree(8292) → 20 studies
Found 200 runs, 23 unique studies
Query: tax_tree(8782) → 20 studies
Found 200 runs, 31 unique studies
Query: tax_tree(7898) → 20 studies
Found 200 runs, 53 unique studies
Query: tax_tree(6656) → 20 studies
Found 200 runs, 21 unique studies
Query: tax_tree(7742) → 20 studies
Found 200 runs, 21 unique studies
Query: tax_tree(40674) → 20 studies

Total unique studies: 112
Processing: PRJDB4069...
 study: PRJDB4069 | host: Aquarana catesbeiana | filters pass: False
Processing: PRJDB90...
 study: PRJDB90 | host: Pelodiscus sinensis japonicus | filters pass: False
Processing: PRJDB3482...
 study: PRJDB3482 | host: Coturnix japonica x Gallus gallus | filters pass: False
Processing: PRJDB12540...
 study: PRJDB12540 | host: Xenopus laevis | filters pass: False
Processing: PRJDA45949...
 study: PRJDA45949 | host: Rattus norvegicus | filters pass: False
Proc

In [ ]:
for study in ["PRJDB11263", "PRJDB5051"]:
    runs_df = fetch_runs_for_study(study)
    host = resolve_host_species(runs_df)
    lib_strategies = runs_df["library_strategy"].value_counts().to_dict()
    provenance = fetch_study_origin(study)
    abstract = fetch_pubmed_abstract(study)
    
    prompt = f"""
Study accession: {study}
Source: {provenance['source']}
Title: {provenance['title']}
Description: {provenance.get('description')}
Resolved host species: {host}
Library strategies: {lib_strategies}

PubMed abstract:
{abstract or 'Not available'}

Please check:
1) Is this a gut microbiome study?
2) Does the title/abstract match the metadata?
3) Any red flags?
Rate consistency: High/Medium/Low
"""

    message = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=512,
        system="You are a scientific quality control assistant checking metadata consistency for microbiome studies. Be direct about whether this is actually a gut microbiome study.",
        messages=[{"role": "user", "content": prompt}]
    )
    
    print(f"\n{'='*50}")
    print(f"Study: {study} | {provenance['title']}")
    print(f"{'='*50}")
    print(message.content[0].text)


Study: PRJDB11263 | Horizontal transfer of BovB LINE from snakes to frogs
## Quality Control Assessment

### 1) Is this a gut microbiome study?
**NO** - This is definitively NOT a gut microbiome study.

This is a molecular evolution study investigating horizontal gene transfer of BovB LINE (Long Interspersed Nuclear Elements) retrotransposons between snakes and frogs. The study focuses on:
- Transposable element phylogenetics
- Horizontal DNA transfer mechanisms
- Parasite-mediated gene transfer between vertebrate species

### 2) Does the title/abstract match the metadata?
**YES** - The metadata is internally consistent:
- Title clearly states "Horizontal transfer of BovB LINE from snakes to frogs"
- Description elaborates on the methodology (PCR amplification, NGS sequencing via PacBio RS II)
- Library strategies align with the approach: AMPLICON (211 samples) for targeted BovB sequencing, plus supplementary RNA-Seq and WGS
- Host species (*Mantidactylus betsileanus* - a Malagasy fro

In [ ]:
import importlib
import src.decisions
importlib.reload(src.decisions)

from src.decisions import save_decisions, load_decisions

# test save
test_decisions = [
    {"study": "PRJDB4366", "host": "Apostichopus japonicus", "decision": "Include", "notes": "Sea cucumber - good phylogenetic gap coverage"},
    {"study": "PRJDB11263", "host": "Mantidactylus betsileanus", "decision": "Exclude", "notes": "Not a microbiome study - BovB LINE transposon research"}
]

save_decisions(test_decisions, "data/decisions_test.json")
loaded = load_decisions("data/decisions_test.json")
print("Saved successfully")

# test load
loaded = load_decisions("data/decisions_test.json")
print("Loaded:", loaded)

Saved successfully
Loaded: [{'study': 'PRJDB4366', 'host': 'Apostichopus japonicus', 'decision': 'Include', 'notes': 'Sea cucumber - good phylogenetic gap coverage'}, {'study': 'PRJDB11263', 'host': 'Mantidactylus betsileanus', 'decision': 'Exclude', 'notes': 'Not a microbiome study - BovB LINE transposon research'}]


In [69]:
import pandas as pd
summary_df = pd.DataFrame(results)
print(summary_df[["study", "host", "has_16S", "has_WGS", "passes"]].to_string())

          study                               host  has_16S  has_WGS  passes
0     PRJDB4069               Aquarana catesbeiana    False    False   False
1       PRJDB90      Pelodiscus sinensis japonicus    False    False   False
2     PRJDB3482  Coturnix japonica x Gallus gallus    False    False   False
3    PRJDB12540                     Xenopus laevis    False    False   False
4    PRJDA45949                  Rattus norvegicus    False    False   False
5     PRJDB1562             Nephotettix cincticeps    False    False   False
6      PRJDB734           Drosophila pseudoobscura    False    False   False
7     PRJDB2790             Cyprichromis leptosoma    False    False   False
8    PRJDA47577                       Mus musculus    False    False   False
9    PRJDA36487                       Mus musculus    False    False   False
10    PRJDB5181                   Penaeus vannamei     True    False   False
11    PRJDB1759                        Bombyx mori    False    False   False